# 6단계 — Dropout 재확인 (계획 3번): rotation 증강과 얼리스탑 아래에서 0.75가 0.5보다 나은가

3단계는 "Dropout 0.5 유지"로 끝났다. 그 결론은 **증강이 없고 5에폭으로 고정한** 조건에서 나왔고, PyTorch의 0.75는 5에폭이 끝날 때 val loss가 아직 내려가는 중이라 5에폭 한계가 섞여 있었다. 그 뒤로 학습 방식(얼리스탑)과 기준선(증강 rotation ±20°)이 바뀌었다. 이 노트북은 같은 질문을 **지금의 기준선에서 한 번 더** 본다: 3단계에서 후보로 남았던 0.75가 rotation 증강과 얼리스탑 아래에서도 0.5보다 나은가.

| | 값 |
|---|---|
| 프레임워크 | Keras, PyTorch |
| 바꾸는 것 | Dropout 비율 **0.5(기준선) → 0.75** (하나만) |
| 고정하는 것 | 증강 rotation ±20° · 얼리스탑(상한 30에폭 · val loss 감시 · patience 5 · 최저 에폭 가중치 복원) · He 초기화 · 2 Conv(32·64) + Dense(11) · Adam 1e-3 · batch 32 · 학습률 고정 |
| 시드 | 0 · 1 · 2 (훑기 없이 바로 3시드. 3단계에서 0.75가 이미 후보로 남았기 때문이다) |
| 실행 횟수 | 새로 돌리는 것은 0.75 줄 3시드 × 2프레임워크 = **6번** |
| 비교 기준 줄 | **Dropout 0.5 · rotation** — 5단계 노트북이 이미 3시드로 돌려 둔 결과(`food11_v2_aug.csv`)를 그대로 가져와 쓴다. 다시 돌리지 않는다 |
| 소요 시간 | 대략 96분 (Keras ≈ 24분, PyTorch ≈ 72분, 어림값). 0.75가 더 늦게 멈추면 길어진다 |

**비교 기준 줄을 다시 돌리지 않아도 되는 이유.** 5단계의 rotation 줄과 이 노트북의 0.75 줄은 코드가 같다. 이 노트북의 Keras·PyTorch 셀은 5단계 셀에서 아래 부분만 바뀌었다. ① Dropout 비율을 인자로 받는다 ② 결과 기록과 "끝난 실행 건너뛰기"의 키에 Dropout 값을 넣는다 ③ 실행 루프가 셀 1의 설정(`CANDS`)을 따른다. 모델 구조 · 초기화 · 옵티마이저 · 얼리스탑 · 증강 · 데이터 불러오기는 그대로다. 같은 시드면 첫 가중치와 데이터 순서, 증강에 쓰는 난수열이 같고 Dropout 비율만 다르다 (GPU 연산 순서 때문에 값이 완전히 같게 재현되지는 않는다).

## Dropout이란

Flatten 바로 뒤에서 학습할 때만 값의 일부(비율 p)를 무작위로 0으로 만드는 정규화다. 특정 값 하나에 기대기 어려워져서 과적합이 줄어든다. 파라미터가 없어서 모델 크기는 그대로다 (653,003개). 검증 · 평가 때는 끈다. 비율이 높을수록 정규화가 세지고, 너무 세면 덜 배운다.

## 예상되는 작용 (가설 — 결과로 확인한다)

- rotation이 이미 같은 사진을 외우기 어렵게 만들어서 val loss 바닥이 11~12에폭까지 늦춰졌다. 그 위에 Dropout을 올려서 **추가로 얻는 것은 3단계(증강 없음)보다 작거나 없을 수 있다**
- 0.75는 학습을 더 늦춘다. 최저 에폭과 멈춘 에폭이 0.5보다 뒤로 밀릴 것이다. 상한 30에폭에 닿으면 표의 비고에 `상한까지 안 멈춤`이 붙는다
- 정규화가 이미 충분한 모델에서 비율을 더 올리면 **덜 배워서 val 정확도가 내려갈 수도 있다**
- 3단계에서는 Keras val loss가 시드 범위에 걸리고 PyTorch val acc는 시드 하나가 반대로 나와서 4칸 중 2칸만 통과했다. 두 프레임워크가 같은 방향이 나올지는 모른다

## 이 단계가 바꾸지 않는 것 (한계)

- **값은 0.5와 0.75 둘만 본다.** rotation이 정규화를 이미 더하므로 더 낮은 값(0.25 등)이 나을 수도 있지만, 이번에는 3단계에서 후보로 남은 0.75만 확인한다
- **증강은 rotation ±20° 한 값이다.** 증강 정책끼리의 비교(계획 5번)는 7단계에서 따로 본다
- 학습률과 He 초기화의 결론은 여전히 5에폭 기준이다 (순차 튜닝의 한계, 상호작용은 못 본다)
- 후보를 고르는 데 val을 쓴다. test는 보고용이다

## 결론을 내는 규칙과 예상 (결과를 보기 전에 정해 둔다)

2~5단계와 같은 규칙이다.

- 0.75 줄을 기준선(0.5) 줄과 **같은 시드끼리** 뺀다. 비교하는 값은 얼리스탑이 돌려준 모델(val loss 최저 에폭)의 **val 정확도와 val loss**다. 시드 전부에서 방향이 같고, 평균 차이가 시드 범위(두 조건의 최대−최소 중 큰 쪽)보다 클 때만 "뚜렷함"이다
- val acc와 val loss **둘 다, 두 프레임워크 모두(4칸 전부)**에서 0.75가 뚜렷하게 낫다면 Dropout을 0.75로 바꾼다. 아니면 0.5를 유지한다
- 4칸 전부에서 0.5가 뚜렷하게 낫다면 "기준선이 뚜렷하게 낫다"로 적는다
- 증강 없는 train 정확도, train − val 격차, test 정확도, 최저 에폭, 멈춘 에폭은 보고용이고 판정에 쓰지 않는다
- 0.75가 채택되면 **7단계(계획 5번)는 Dropout 0.75에서 돌린다** (7단계 셀 1의 `BASE_DROPOUT`). 0.5가 유지되면 7단계는 0.5 그대로다

## 실행 방법

위에서부터 셀을 차례로 돌린다 (셀 0 → 1 → 2 → 3 → 4). 자리를 비울 때는 셀 0에서 Drive 허용을 누른 뒤 "런타임 → 모두 실행"으로 이어서 돌려도 된다. 끝나면 셀 4가 요약을 낸다.

- **GPU(T4)로 돌린다.** 이전 단계가 전부 T4였다
- **5단계 결과 파일(`food11_v2_aug.csv`)이 같은 Drive(내 드라이브 맨 위)에 있어야 한다.** 비교 기준 줄을 거기서 가져온다. 셀 1의 `[계획]`에 기준선 줄이 시드 0·1·2 모두 있는지 나온다 (`⚠ 부족`이 보이면 판정할 수 없다)
- 셀 1의 `[계획]`에 남은 실행 수와 시간 어림이 나온다. 이 노트북의 결과는 Drive의 `food11_v2_dropout_recheck.csv`에 실행 하나가 끝날 때마다 한 줄씩 쌓인다. 다시 돌리면 끝난 실행은 건너뛰고 이어서 한다 (런타임이 끊기면 그 실행만 처음부터 다시 돈다). 처음부터 다시 하려면 그 파일을 지운다
- 셀 2(Keras)와 셀 3(PyTorch)은 서로 독립이다. 셀 4(요약)는 있는 결과만으로 만든다
- 실행 하나마다 시작 줄 → 에폭 줄 → 끝 줄이 나온다. 에폭 줄 끝에 `최저 갱신` 또는 `대기 n/5`가 붙고, 대기가 5가 되면 멈춘다. 몇 에폭에서 멈출지는 실행마다 다르다
- **셀 4의 출력을 통째로 붙여 주면 된다.**

## 어떻게 읽을까

셀 4는 출력과 그래프 2개를 만든다: `[결과표]`(프레임워크마다) · `[줄의 출처]` · `[기준선 대비]` · 판정 줄 · 결론 줄 · 그래프 `dropout_recheck_curves.png`(에폭별 곡선)와 `dropout_recheck_seeds.png`(줄별 결과).

- **결과표**의 값은 얼리스탑이 돌려준 모델(val loss 최저 에폭)의 값이다 (평균 [최소~최대]). train 정확도는 증강 없이 다시 잰 값이다. 최저 에폭은 모델을 돌려받은 에폭이고, 비고의 `상한까지 안 멈춤`은 30에폭이 끝날 때까지 val loss가 내려가는 중이었다는 뜻이다 (상한이 걸림돌일 수 있다)
- **줄의 출처**: Dropout 0.5 줄은 5단계 결과를 가져온 것이다. 어느 파일의 어느 시드인지 나온다
- **기준선 대비**: 시드 3개가 있으면 위 규칙대로 판정 줄이 나온다. 결론 줄이 7단계를 어떤 Dropout으로 돌릴지까지 알려 준다
- 곡선 그래프: 줄마다 색이 하나다 (점선 = 기준선). 연한 선은 시드 하나, 굵은 선은 평균이고, 표시(세모 · 역세모)는 얼리스탑이 돌려준 에폭이다
- 줄별 그래프: 점 하나가 시드 하나이고 같은 시드는 같은 가로 위치에 있다 (왼쪽부터 시드 0 · 1 · 2). 짧은 가로선이 평균이다

## 셀 0 — 데이터 복사 (최초 1회, 런타임 끊기면 다시)

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

# Drive에서 직접 읽으면 학습이 수십 배 느려져서 로컬로 복사한다
if not os.path.isdir("/content/food11"):
    !cp -r /content/drive/MyDrive/food11 /content/food11

## 셀 1 — 공통 설정, 증강 정의, 얼리스탑 규칙, 기록 함수, 실행 계획 (여기서 값을 바꾼다)

In [ ]:
import csv
import gc
import json
import os
import time

import numpy as np

DATA_DIR = "/content/food11"
IMG, BATCH = 128, 32
NUM_CLASSES = 11
N_PARAMS = 653_003           # 두 프레임워크 모두 이 값이어야 같은 모델이다 (증강 층은 파라미터가 없다)

# 5단계까지의 기준선 (이 노트북에서 바꾸지 않는다): He 초기화, 2 Conv(32·64) + Dropout + Dense(11),
# Adam 1e-3, batch 32, 학습률 고정, 얼리스탑. 증강과 Dropout 값만 아래 '이 노트북이 보는 것'에서 정한다.
OPTIMIZER, LR = "adam", 1e-3
EPOCH_CAP = 30               # 얼리스탑: 상한
PATIENCE = 5                 # val loss가 최저를 갱신하지 못한 에폭이 5번 쌓이면 멈추고, 최저였던 에폭의 가중치로 되돌린다
ROTATION_DEG = 20            # 회전: ±20도
ZOOM = 0.2                   # 확대: 가운데 기준 1.0 ~ 1/(1-0.2) = 1.25배 (확대만이라 빈 곳 채움이 필요 없다)

# ============ 이 노트북이 보는 것: 계획 3번 — rotation 증강 아래에서 Dropout 재확인 ============
SEEDS = (0, 1, 2)                          # 기준선 줄이 3시드(0·1·2)라 같은 시드로 맞춘다
BASE_AUG, BASE_DROPOUT = "rotation", 0.5   # 기준선 줄 — 5단계 결과를 가져와서 쓴다 (다시 돌리지 않는다)
DROPOUTS = (0.75,)                         # 이번에 새로 돌리는 Dropout 값 (하나)
REF = (BASE_AUG, BASE_DROPOUT)
CANDS = tuple((BASE_AUG, d) for d in DROPOUTS)
CONTEXT = ()                               # 판정에 쓰지 않고 같이 보여 주기만 하는 줄
LABELS = {c: f"Dropout {c[1]}" for c in (REF, *CANDS, *CONTEXT)}
MIN_PER_RUN = {"keras": 8, "torch": 24}    # 새로 돌리는 실행 하나에 걸리는 시간 (분, 어림값). 몇 에폭에서 멈추느냐에 따라 달라진다

# 실행 하나가 끝날 때마다 한 줄씩 쌓는다. 런타임이 끊겨도 남고, 다시 돌리면 끝난 실행은 건너뛴다.
# 처음부터 다시 하려면 이 파일을 지운다.
RESULTS = "/content/drive/MyDrive/food11_v2_dropout_recheck.csv"
REF_FILES = ["/content/drive/MyDrive/food11_v2_aug.csv"]       # 기준선 줄을 가져오는 곳 (5단계 결과, 읽기만 한다)

# 요약 셀의 표 · 그래프 · 결론 문구
PLAN_TITLE = "계획 3번 — Dropout 재확인 (rotation 증강 아래, 얼리스탑)"
COL_HEAD = "Dropout"
TABLE_NOTE = "rotation 증강, Adam 1e-3"
STYLE = {LABELS[REF]: ("#d55e00", "^"), LABELS[CANDS[0]]: ("#a01020", "v")}    # 줄 이름 -> (색, 표시)
FIG_PREFIX = "dropout_recheck"
FIG_TITLE = "Dropout recheck under rotation augmentation, Adam 1e-3, He init, early stopping"
X_LABEL = "dropout (rotation augmentation on)"
ADOPT = ("-> 결론: Dropout을 {pick_val}로 바꾼다 (4칸 전부에서 {base}보다 뚜렷하게 앞섰다)\n"
         "   다음: 7단계 노트북(계획 5번)은 셀 1의 BASE_DROPOUT을 {pick_val}로 바꿔서 돌린다")
KEEP = ("-> 결론: Dropout {base_val} 유지 — 새로 돌린 값({pick})이 4칸 전부에서 뚜렷하게 앞서지 않았다\n"
        "   다음: 7단계 노트북(계획 5번)은 BASE_DROPOUT {base_val} 그대로 돌린다")

FIELDS = ["framework", "arm", "optimizer", "lr", "dropout", "seed", "n_params", "epoch_cap", "patience",
          "minutes", "stop_epoch", "best_epoch",
          "train_acc_curve", "train_loss_curve", "val_acc_curve", "val_loss_curve",    # 에폭별 값 (JSON 리스트, 멈춘 에폭까지)
          "val_acc", "val_loss", "test_acc", "test_loss", "train_clean_acc"]           # 돌려받은 모델(최저 에폭)의 값
FWN = (("keras", "Keras"), ("torch", "PyTorch"))
assert len(CANDS) == 1 and REF not in CANDS, "이 노트북은 새로 돌리는 줄 하나를 기준선 줄과 비교한다"
assert len(set(LABELS.values())) == len(LABELS), "줄 이름이 겹친다"


def keras_aug(arm, seed):
    """Keras 증강 층. 학습할 때만 켜지고 평가에서는 입력을 그대로 내보낸다. 빈 곳은 검정, 보간은 bilinear.
    seed를 직접 넘겨서, 증강 층을 만들어도 뒤따르는 층의 초기화 난수가 none 줄과 같게 한다.
    all은 flip -> rotation -> zoom을 차례로 모두 켠다. 층마다 시드를 다르게 준다: 같은 시드를 주면 세 층이 같은 난수열을 써서
    반전 · 회전 · 확대가 서로 묶인다 (합성 이미지로 확인: 같은 시드 상관 0.88~0.99, 다른 시드 0.04 이하)."""
    from keras import layers
    if arm == "all":
        return keras_aug("flip", seed) + keras_aug("rotation", seed + 100) + keras_aug("zoom", seed + 200)
    if arm == "flip":
        return [layers.RandomFlip("horizontal", seed=seed)]
    if arm == "rotation":
        return [layers.RandomRotation(ROTATION_DEG / 360.0, fill_mode="constant", interpolation="bilinear",
                                      fill_value=0.0, seed=seed)]
    if arm == "zoom":
        return [layers.RandomZoom((-ZOOM, 0.0), fill_mode="constant", interpolation="bilinear",
                                  fill_value=0.0, seed=seed)]
    assert arm == "none", arm
    return []


def torch_aug(arm):
    """PyTorch 증강 변환 (128x128로 줄인 PIL 이미지에 건다). Keras와 같은 범위 · 보간(bilinear) · 채움(검정 0).
    all은 flip -> rotation -> zoom을 차례로 모두 켠다 (변환마다 따로 무작위로 뽑는다)."""
    from torchvision import transforms
    from torchvision.transforms import InterpolationMode
    if arm == "all":
        return torch_aug("flip") + torch_aug("rotation") + torch_aug("zoom")
    if arm == "flip":
        return [transforms.RandomHorizontalFlip(0.5)]
    if arm == "rotation":
        return [transforms.RandomRotation(ROTATION_DEG, interpolation=InterpolationMode.BILINEAR, fill=0)]
    if arm == "zoom":
        return [transforms.RandomAffine(0, scale=(1.0, 1.0 / (1.0 - ZOOM)),
                                        interpolation=InterpolationMode.BILINEAR, fill=0)]
    assert arm == "none", arm
    return []


class Stopper:
    """얼리스탑 규칙 (두 프레임워크 공통). Keras EarlyStopping(monitor="val_loss", min_delta=0)과 같은 규칙이다.
    val loss가 지금까지의 최저보다 낮아지면 최저를 갱신하고, 아니면 기다린 횟수(wait)를 하나 올린다.
    wait가 patience에 닿으면 멈춘다. 되돌릴 가중치는 호출하는 쪽이 최저를 갱신할 때마다 담아 둔다."""

    def __init__(self, patience):
        self.patience, self.best, self.best_epoch, self.wait = patience, float("inf"), 0, 0

    def update(self, epoch, val_loss):
        """epoch는 1부터 센다. 반환: (최저를 갱신했는가, 멈춰야 하는가)"""
        if val_loss < self.best:
            self.best, self.best_epoch, self.wait = val_loss, epoch, 0
            return True, False
        self.wait += 1
        return False, self.wait >= self.patience


def epoch_line(epoch, train_acc, train_loss, val_acc, val_loss, stopper, t0):
    if stopper.wait == 0:
        status = "최저 갱신"
    else:
        status = f"대기 {stopper.wait}/{stopper.patience}" + (" → 멈춤" if stopper.wait >= stopper.patience else "")
    return (f"  에폭 {epoch}  train {train_acc:.3f} (loss {train_loss:.3f})  ·  "
            f"val {val_acc:.3f} (loss {val_loss:.3f})  ·  {status}  ·  {(time.time() - t0) / 60:.1f}분")


def usable(r):
    """같은 학습 방식(옵티마이저 · 학습률 · 얼리스탑)의 결과만 쓴다 (설정을 바꾸고 다시 돌려도 예전 결과와 섞이지 않는다)."""
    return (r["optimizer"] == OPTIMIZER and float(r["lr"]) == LR and int(r["epoch_cap"]) == EPOCH_CAP
            and int(r["patience"]) == PATIENCE)


def load_all():
    """REF_FILES(가져다 쓰는 결과)와 RESULTS(이 노트북 결과)를 모은다: (프레임워크, 증강, Dropout, 시드) -> 한 줄.
    같은 키가 여럿이면 나중에 읽은 쪽이 이긴다 (RESULTS가 마지막)."""
    got = {}
    for path in [*REF_FILES, RESULTS]:
        if not os.path.exists(path):
            continue
        with open(path, newline="", encoding="utf-8") as f:
            for r in csv.DictReader(f):
                if usable(r):
                    got[(r["framework"], r["arm"], float(r["dropout"]), int(r["seed"]))] = dict(r, _src=os.path.basename(path))
    return got


def done_runs():
    """이 노트북이 이미 끝낸 실행 (RESULTS 파일만 본다)."""
    if not os.path.exists(RESULTS):
        return set()
    with open(RESULTS, newline="", encoding="utf-8") as f:
        return {(r["framework"], r["arm"], float(r["dropout"]), int(r["seed"])) for r in csv.DictReader(f) if usable(r)}


def save_row(row):
    is_new = not os.path.exists(RESULTS)
    with open(RESULTS, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if is_new:
            w.writeheader()
        w.writerow(row)


def make_row(framework, arm, dropout, seed, n_params, minutes, stopper, hist, final):
    """hist: 에폭별 리스트 4개(멈춘 에폭까지), final: 되돌린 모델의 val · test · 증강 없는 train 값"""
    return dict(framework=framework, arm=arm, optimizer=OPTIMIZER, lr=LR, dropout=dropout, seed=seed,
                n_params=n_params, epoch_cap=EPOCH_CAP, patience=PATIENCE, minutes=round(minutes, 2),
                stop_epoch=len(hist["val_loss"]), best_epoch=stopper.best_epoch,
                train_acc_curve=json.dumps(hist["train_acc"]), train_loss_curve=json.dumps(hist["train_loss"]),
                val_acc_curve=json.dumps(hist["val_acc"]), val_loss_curve=json.dumps(hist["val_loss"]),
                val_acc=final["val_acc"], val_loss=final["val_loss"], test_acc=final["test_acc"],
                test_loss=final["test_loss"], train_clean_acc=final["train_clean_acc"])


def tag(framework, arm, dropout, seed):
    return f"[{framework} · {arm} · Dropout {dropout} · 시드 {seed}]"


def report(row):
    loss = float(row["val_loss"])
    print(f"{tag(row['framework'], row['arm'], float(row['dropout']), row['seed'])} 최저 에폭 {row['best_epoch']} · "
          f"{row['stop_epoch']}에폭에서 멈춤 · val {float(row['val_acc']):.3f} (loss {loss:.2f}) · "
          f"test {float(row['test_acc']):.3f} · {row['minutes']:.1f}분"
          + ("" if np.isfinite(loss) else " · 발산(loss NaN)"))


def cond_text(c):
    return f"{c[0]} · Dropout {c[1]}"


def plan():
    done, known = done_runs(), load_all()
    print(f"[계획] {PLAN_TITLE}")
    print(f"  새로 돌리는 줄: {cond_text(CANDS[0])} × 시드 {', '.join(map(str, SEEDS))} — "
          f"얼리스탑(상한 {EPOCH_CAP}에폭, patience {PATIENCE}), CSV에 이미 있는 실행은 건너뛴다")
    missing = False
    for role, conds in (("기준선 줄", (REF,)), ("참고 줄", CONTEXT)):
        for c in conds:
            for fw, name in FWN:
                have = [s for s in SEEDS if (fw, c[0], c[1], s) in known]
                files = sorted({known[(fw, c[0], c[1], s)]["_src"] for s in have})
                ok = len(have) == len(SEEDS)
                missing = missing or (role == "기준선 줄" and not ok)
                print(f"  {role} {cond_text(c)} · {name:8s} 시드 {', '.join(map(str, have)) or '없음'}"
                      + (" ✓" if ok else " ⚠ 부족") + (f"  ({', '.join(files)}에서 가져옴)" if files else ""))
    if missing:
        print(f"  ⚠ 기준선 줄이 없거나 모자란다 — 5단계 결과 파일을 Drive에 두었는지 확인할 것: {', '.join(REF_FILES)}")
    for fw, name in FWN:
        todo = [(c, s) for s in SEEDS for c in CANDS if (fw, c[0], c[1], s) not in done]
        total = len(SEEDS) * len(CANDS)
        print(f"  {name:8s} 이미 {total - len(todo)}번 · 남은 {len(todo)}번 ≈ {int(len(todo) * MIN_PER_RUN[fw] + 0.5)}분")


plan()

## 셀 2 — Keras (rotation 증강 · Dropout 0.75, 얼리스탑) · 시드 3개에 약 24분

In [ ]:
import keras
from keras import layers


def make_optimizer(name, lr):
    if name == "adam":
        return keras.optimizers.Adam(learning_rate=lr)
    if name == "sgd_momentum":
        return keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
    return keras.optimizers.SGD(learning_rate=lr)


def run_keras(arm, dropout, seed):
    keras.utils.set_random_seed(seed)

    def load(split, shuffle):
        ds = keras.utils.image_dataset_from_directory(
            f"{DATA_DIR}/{split}", image_size=(IMG, IMG),
            batch_size=BATCH, shuffle=shuffle, seed=seed)
        rescale = layers.Rescaling(1.0 / 255)
        return ds.map(lambda x, y: (rescale(x), y))

    train_ds = load("training", True)
    train_eval_ds = load("training", False)       # 증강 없이 train 정확도를 다시 재는 용도
    val_ds = load("validation", False)
    test_ds = load("evaluation", False)

    # 128 -> conv 126 -> pool 63 -> conv 61 -> pool 30
    model = keras.Sequential([
        keras.Input(shape=(IMG, IMG, 3)),
        *keras_aug(arm, seed),                    # 학습할 때만 켜지는 증강 층 (none이면 없다)

        layers.Conv2D(32, 3, activation="relu", kernel_initializer="he_normal"),
        layers.MaxPooling2D(2),

        layers.Conv2D(64, 3, activation="relu", kernel_initializer="he_normal"),
        layers.MaxPooling2D(2),

        layers.Flatten(),
        layers.Dropout(dropout),
        layers.Dense(NUM_CLASSES, activation="softmax",
                     kernel_initializer="he_normal"),
    ])
    n_params = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    assert n_params == N_PARAMS, n_params

    model.compile(
        optimizer=make_optimizer(OPTIMIZER, LR),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    stopper, best = Stopper(PATIENCE), {}
    t0 = time.time()

    def on_epoch(epoch, logs):                    # 에폭이 끝날 때마다: 최저를 갱신하면 가중치를 담고, 멈출 때가 되면 멈춘다
        improved, stop = stopper.update(epoch + 1, logs["val_loss"])
        if improved:
            best["weights"] = model.get_weights()
        if stop:
            model.stop_training = True
        print(epoch_line(epoch + 1, logs["accuracy"], logs["loss"], logs["val_accuracy"],
                         logs["val_loss"], stopper, t0), flush=True)

    h = model.fit(train_ds, validation_data=val_ds, epochs=EPOCH_CAP, verbose=0,
                  callbacks=[keras.callbacks.LambdaCallback(on_epoch_end=on_epoch)]).history
    if "weights" in best:
        model.set_weights(best["weights"])        # 최저였던 에폭의 가중치로 되돌린다
    val_loss, val_acc = model.evaluate(val_ds, verbose=0)
    test_loss, test_acc = model.evaluate(test_ds, verbose=0)
    _, train_clean = model.evaluate(train_eval_ds, verbose=0)
    hist = {"train_acc": h["accuracy"], "train_loss": h["loss"],
            "val_acc": h["val_accuracy"], "val_loss": h["val_loss"]}
    row = make_row("keras", arm, dropout, seed, n_params, (time.time() - t0) / 60, stopper,
                   {k: [float(v) for v in vs] for k, vs in hist.items()},
                   dict(val_loss=float(val_loss), val_acc=float(val_acc), test_loss=float(test_loss),
                        test_acc=float(test_acc), train_clean_acc=float(train_clean)))
    keras.backend.clear_session()
    gc.collect()
    return row


for seed in SEEDS:
    for arm, dropout in CANDS:
        if ("keras", arm, dropout, seed) in done_runs():
            print(f"{tag('keras', arm, dropout, seed)} 이미 있음 — 건너뜀")
            continue
        print(f"{tag('keras', arm, dropout, seed)} 시작 (약 {MIN_PER_RUN['keras']}분)", flush=True)
        row = run_keras(arm, dropout, seed)
        save_row(row)
        report(row)

## 셀 3 — PyTorch (rotation 증강 · Dropout 0.75, 얼리스탑) · 시드 3개에 약 72분

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

tf_ = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),          # 0~1 변환 + HWC -> CHW
])


def make_optimizer(name, params, lr):
    if name == "adam":
        return torch.optim.Adam(params, lr=lr)
    if name == "sgd_momentum":
        return torch.optim.SGD(params, lr=lr, momentum=0.9)
    return torch.optim.SGD(params, lr=lr)


def run_torch(arm, dropout, seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    g = torch.Generator().manual_seed(seed)

    # 증강은 train에만 건다. 128x128로 줄인 뒤에 걸어서 Keras와 같은 해상도에서 변형한다 (none이면 tf_와 같다)
    train_tf = transforms.Compose([transforms.Resize((IMG, IMG)), *torch_aug(arm), transforms.ToTensor()])

    def load(split, shuffle, tf=tf_):
        ds = datasets.ImageFolder(f"{DATA_DIR}/{split}", transform=tf)
        return DataLoader(ds, batch_size=BATCH, shuffle=shuffle,
                          generator=g if shuffle else None,
                          num_workers=2, pin_memory=True)

    train_dl = load("training", True, train_tf)
    train_eval_dl = load("training", False)       # 증강 없이 train 정확도를 다시 재는 용도
    val_dl = load("validation", False)
    test_dl = load("evaluation", False)

    # 128 -> conv 126 -> pool 63 -> conv 61 -> pool 30
    model = nn.Sequential(
        nn.Conv2d(3, 32, 3), nn.ReLU(), nn.MaxPool2d(2, 2),
        nn.Conv2d(32, 64, 3), nn.ReLU(), nn.MaxPool2d(2, 2),
        nn.Flatten(),
        nn.Dropout(dropout),
        nn.Linear(64 * 30 * 30, NUM_CLASSES),
    ).to(device)

    for m in model:       # 1단계 — He 초기화. 가중치만, bias는 PyTorch 기본값 그대로
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="relu")

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert n_params == N_PARAMS, n_params

    loss_fn = nn.CrossEntropyLoss()
    optimizer = make_optimizer(OPTIMIZER, model.parameters(), LR)

    def evaluate(dataloader):
        model.eval()
        total, correct, n = 0.0, 0, 0
        with torch.no_grad():
            for X, y in dataloader:
                X, y = X.to(device), y.to(device)
                pred = model(X)
                total += loss_fn(pred, y).item() * y.size(0)
                correct += (pred.argmax(1) == y).sum().item()
                n += y.size(0)
        return total / n, correct / n

    hist = {"train_acc": [], "train_loss": [], "val_acc": [], "val_loss": []}
    stopper, best_state = Stopper(PATIENCE), None
    t0 = time.time()
    for epoch in range(EPOCH_CAP):
        model.train()
        total, correct, n = 0.0, 0, 0
        for X, y in train_dl:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            loss = loss_fn(pred, y)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total += loss.item() * y.size(0)
            correct += (pred.argmax(1) == y).sum().item()
            n += y.size(0)
        val_loss, val_acc = evaluate(val_dl)
        hist["train_acc"].append(correct / n)
        hist["train_loss"].append(total / n)
        hist["val_acc"].append(val_acc)
        hist["val_loss"].append(val_loss)
        improved, stop = stopper.update(epoch + 1, val_loss)
        if improved:                              # 최저를 갱신하면 가중치를 담아 둔다
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        print(epoch_line(epoch + 1, hist["train_acc"][-1], hist["train_loss"][-1], val_acc, val_loss,
                         stopper, t0), flush=True)
        if stop:
            break

    if best_state is not None:
        model.load_state_dict(best_state)         # 최저였던 에폭의 가중치로 되돌린다
    val_loss, val_acc = evaluate(val_dl)
    test_loss, test_acc = evaluate(test_dl)
    _, train_clean = evaluate(train_eval_dl)
    row = make_row("torch", arm, dropout, seed, n_params, (time.time() - t0) / 60, stopper, hist,
                   dict(val_loss=val_loss, val_acc=val_acc, test_loss=test_loss,
                        test_acc=test_acc, train_clean_acc=train_clean))
    del model, optimizer, train_dl, train_eval_dl, val_dl, test_dl
    torch.cuda.empty_cache()
    gc.collect()
    return row


for seed in SEEDS:
    for arm, dropout in CANDS:
        if ("torch", arm, dropout, seed) in done_runs():
            print(f"{tag('torch', arm, dropout, seed)} 이미 있음 — 건너뜀")
            continue
        print(f"{tag('torch', arm, dropout, seed)} 시작 (약 {MIN_PER_RUN['torch']}분)", flush=True)
        row = run_torch(arm, dropout, seed)
        save_row(row)
        report(row)

## 셀 4 — 요약과 판정 (이 셀의 출력을 붙여 주면 된다)

In [ ]:
import unicodedata

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

known = load_all()
rows = {k: r for k, r in known.items() if (k[1], k[2]) in LABELS}
assert any((k[1], k[2]) in CANDS for k in rows), "이 노트북의 결과가 아직 없다 — 셀 2·3을 먼저 돌릴 것"
assert all(int(r["n_params"]) == N_PARAMS for r in rows.values()), "파라미터 수가 다른 실행이 있다"

FW = {"keras": "Keras", "torch": "PyTorch"}
BASE_LAB, CAND_LAB = LABELS[REF], LABELS[CANDS[0]]
CTX_LABS = [LABELS[c] for c in CONTEXT]
MIN_SEEDS = 3                  # 규칙을 적용하려면 조건마다 이만큼의 시드가 필요하다


def width(s):
    return sum(2 if unicodedata.east_asian_width(c) in "WF" else 1 for c in s)


def pad(s, w):
    return s + " " * max(0, w - width(s))


def bar(title):
    print()
    print("=" * 100)
    print(title)
    print("=" * 100)


def metrics(r):
    """얼리스탑이 돌려준 모델(최저 에폭)의 값."""
    loss = float(r["val_loss"])
    best, stop = int(r["best_epoch"]), int(r["stop_epoch"])
    val_acc, train = float(r["val_acc"]) * 100, float(r["train_clean_acc"]) * 100
    return dict(train_acc=train, val_acc=val_acc, gap=train - val_acc, val_loss=loss,
                best_ep=best, stop_ep=stop, test_acc=float(r["test_acc"]) * 100,
                capped=bool(stop >= EPOCH_CAP and EPOCH_CAP - best < PATIENCE),
                diverged=bool(best == 0 or not np.isfinite(loss)))


runs = {}      # (프레임워크, 줄 이름) -> {시드: 지표}
curves = {}    # (프레임워크, 줄 이름) -> {시드: 에폭별 곡선}
source = {}    # (프레임워크, 줄 이름) -> {시드: 가져온 파일}
for (fw, arm, d, s), r in rows.items():
    k = (fw, LABELS[(arm, d)])
    runs.setdefault(k, {})[s] = metrics(r)
    curves.setdefault(k, {})[s] = dict(val_loss=np.array(json.loads(r["val_loss_curve"]), dtype=float),
                                       val_acc=np.array(json.loads(r["val_acc_curve"]), dtype=float) * 100,
                                       best=int(r["best_epoch"]))
    source.setdefault(k, {})[s] = r["_src"]

fws = [fw for fw in FW if any(k[0] == fw for k in runs)]
arms = [a for a in (*CTX_LABS, BASE_LAB, CAND_LAB) if any((fw, a) in runs for fw in fws)]
seeds_all = sorted({s for g in runs.values() for s in g})
if BASE_LAB not in arms:
    print(f"⚠ 기준선 줄({BASE_LAB})의 결과가 없다 — 5단계 결과 파일을 확인할 것: {', '.join(REF_FILES)}. 판정할 수 없다")


def vals(group, key):
    v = np.array([m[key] for m in group.values()], dtype=float)
    return v[np.isfinite(v)]


def span(group, key, nd):
    v = vals(group, key)
    if len(v) == 0:
        return "-"
    if key.endswith("_ep") and len(group) == 1:
        nd = 0
    if len(group) == 1:
        return f"{v[0]:.{nd}f}"
    return f"{v.mean():.{nd}f} [{v.min():.{nd}f}~{v.max():.{nd}f}]"


def notes(a, group):
    out = []
    if a == BASE_LAB:
        out.append("기준선")
    elif a in CTX_LABS:
        out.append("참고 (판정 제외)")
    n_div = sum(m["diverged"] for m in group.values())
    if n_div:
        out.append(f"발산 {n_div}/{len(group)}")
    n_cap = sum(m["capped"] for m in group.values())
    if n_cap == len(group):
        out.append("상한까지 안 멈춤")
    elif n_cap:
        out.append(f"상한까지 안 멈춤 {n_cap}/{len(group)}")
    return " · ".join(out)


# ---- 결과표 -----------------------------------------------------------------
COLS = [("train_acc", "train acc %", 1), ("val_acc", "val acc %", 1), ("gap", "train − val %p", 1),
        ("val_loss", "val loss 최저", 2), ("best_ep", "최저 에폭", 1), ("stop_ep", "멈춘 에폭", 1),
        ("test_acc", "test acc %", 1)]
for fw in fws:
    seeds_fw = sorted({s for a in arms for s in runs.get((fw, a), {})})
    bar(f"[결과표] {FW[fw]} — {TABLE_NOTE}, 얼리스탑(상한 {EPOCH_CAP}, patience {PATIENCE}), "
        f"시드 {', '.join(map(str, seeds_fw))}" + ("" if len(seeds_fw) == 1 else ", 평균 [최소~최대]"))
    print("   값은 얼리스탑이 돌려준 모델(val loss 최저 에폭)의 것이다. train acc는 증강 없이 다시 잰 값이다")
    head = [COL_HEAD, "시드"] + [name for _, name, _ in COLS] + ["비고"]
    body = []
    for a in arms:
        g = runs.get((fw, a))
        if not g:
            body.append([a, "결과 없음"] + [""] * (len(head) - 2))
            continue
        body.append([a, ",".join(map(str, sorted(g)))]
                    + [span(g, key, nd) for key, _, nd in COLS] + [notes(a, g)])
    ws = [max(width(r[i]) for r in [head] + body) + 2 for i in range(len(head) - 1)]
    for r in [head] + body:
        print("".join(pad(c, w) for c, w in zip(r[:-1], ws)) + r[-1])

# ---- 줄의 출처 --------------------------------------------------------------
bar("[줄의 출처] 기준선 · 참고 줄은 이미 있는 결과를 가져온 것이다 (이 노트북에서 다시 돌리지 않았다)")
for a in arms:
    for fw in fws:
        src = source.get((fw, a))
        if not src:
            continue
        by_file = {}
        for s, f in sorted(src.items()):
            by_file.setdefault(f, []).append(s)
        print("   " + pad(a, 22) + pad(FW[fw], 9) + " · ".join(f"{f} (시드 {', '.join(map(str, ss))})" for f, ss in by_file.items()))


# ---- 같은 시드끼리 비교 -----------------------------------------------------
def paired(fw, a, b, key, higher_better):
    """같은 시드끼리 뺀 값 a-b로 줄 a와 b를 비교한다. higher_better=False면 값이 낮은 쪽이 낫다."""
    ga, gb = runs.get((fw, a), {}), runs.get((fw, b), {})
    seeds = [s for s in sorted(set(ga) & set(gb))
             if np.isfinite(ga[s][key]) and np.isfinite(gb[s][key])]
    out = dict(diffs=[], mean=np.nan, limit=np.nan, winner=None, enough=len(seeds) >= MIN_SEEDS,
               text="비교할 공통 시드가 없다")
    if not seeds:
        return out
    va = np.array([ga[s][key] for s in seeds])
    vb = np.array([gb[s][key] for s in seeds])
    d = va - vb
    out.update(diffs=[float(x) for x in d], mean=float(d.mean()))
    if not out["enough"]:
        out["text"] = f"판정 안 함 (공통 시드 {len(seeds)}개, {MIN_SEEDS}개 필요)"
        return out
    gain = d if higher_better else -d              # 양수 = a가 낫다
    limit = float(max(np.ptp(va), np.ptp(vb)))      # 시드 범위: 두 조건 중 큰 쪽
    out["limit"] = limit
    consistent = bool((gain > 0).all() or (gain < 0).all())
    if consistent and abs(float(gain.mean())) > limit:
        winner = "a" if gain.mean() > 0 else "b"
        out["winner"] = winner
        out["text"] = f"{a if winner == 'a' else b} 우세 (뚜렷함)"
    elif not consistent:
        out["text"] = (f"구분 안 됨 — 시드마다 방향이 다름 "
                       f"({a} 우세 {int((gain > 0).sum())}/{len(seeds)})")
    else:
        out["text"] = "구분 안 됨 — 방향은 같지만 평균 차이가 시드 범위 안"
    return out


def show_cmp(fw, a, b, key, higher_better, title, nd, unit, judge=True):
    c = paired(fw, a, b, key, higher_better)
    head = pad(FW[fw], 9) + pad(a, 14) + pad(title, 18)
    if not c["diffs"]:
        print(head + c["text"])
        return c
    per = " ".join(f"{x:+.{nd}f}" for x in c["diffs"])
    lim = f"  시드 범위 {c['limit']:.{nd}f}" if c["enough"] else ""
    tail = f"  -> {c['text']}" if judge else "  (보고용)"
    print(head + f"{c['mean']:+.{nd}f}{unit}  (시드별 {per}){lim}" + tail)
    return c


bar(f"[기준선 대비] 같은 시드끼리 뺀 값 ({CAND_LAB} − {BASE_LAB}), val loss는 낮은 쪽이 낫다\n"
    "   뚜렷함 = 시드 전부 같은 방향이고 평균 차이가 시드 범위(두 조건의 최대−최소 중 큰 쪽)보다 클 것\n"
    "   시드가 3개 미만이면 판정하지 않고 차이만 보여 준다 (훑어보기)")
cells = []
for fw in fws:
    cells.append(show_cmp(fw, CAND_LAB, BASE_LAB, "val_acc", True, "val acc", 1, "%p"))
    cells.append(show_cmp(fw, CAND_LAB, BASE_LAB, "val_loss", False, "val loss 최저", 3, ""))
    show_cmp(fw, CAND_LAB, BASE_LAB, "gap", False, "train − val 격차", 1, "%p", judge=False)
    show_cmp(fw, CAND_LAB, BASE_LAB, "test_acc", True, "test acc", 1, "%p", judge=False)
    show_cmp(fw, CAND_LAB, BASE_LAB, "best_ep", True, "최저 에폭", 1, "에폭", judge=False)

for x in CTX_LABS:
    bar(f"[참고] {x} 대비 (줄 − {x}) — 보고용이고 판정에 쓰지 않는다")
    if x not in arms:
        print(f"   {x} 줄의 결과가 없다")
        continue
    for a in (BASE_LAB, CAND_LAB):
        for fw in fws:
            show_cmp(fw, a, x, "val_acc", True, "val acc", 1, "%p", judge=False)
            show_cmp(fw, a, x, "val_loss", False, "val loss 최저", 3, "", judge=False)

# ---- 판정 · 결론 -------------------------------------------------------------
print()
print(f"-> 판정 (4칸 = val acc · val loss 최저 × 두 프레임워크, {BASE_LAB} 대비)")
n_new = sum(c["winner"] == "a" for c in cells)
n_base = sum(c["winner"] == "b" for c in cells)
n_skip = sum(not c["enough"] for c in cells)
n_tie = len(cells) - n_new - n_base - n_skip
movers = []
hold = len(fws) < 2 or n_skip > 0
if hold:
    label = "판정 보류"
elif n_new == len(cells):
    label = "기준선보다 뚜렷하게 낫다 -> 기준선에 넣을 후보"
    movers.append(CAND_LAB)
elif n_base == len(cells):
    label = "기준선이 뚜렷하게 낫다"
else:
    label = "구분 안 됨"
extra = f" · 판정 안 함 {n_skip}칸" if n_skip else ""
print(f"   {CAND_LAB}: {label}  ({CAND_LAB} 우세 {n_new}칸 · 기준선 우세 {n_base}칸 · 구분 안 됨 {n_tie}칸{extra})")
print()
fmt = dict(pick=CAND_LAB, base=BASE_LAB, pick_val=CANDS[0][1], base_val=REF[1])
if hold:
    print("-> 결론: 판정 보류 — 두 프레임워크 모두 새로 돌린 줄과 기준선 줄이 시드 3개씩 필요하다. 지금은 큰 차이만 읽는다")
elif movers:
    print(ADOPT.format(**fmt))
else:
    print(KEEP.format(**fmt))

# ---- 그래프 -----------------------------------------------------------------
SURFACE, INK, INK2, MUTED, GRID, AXIS = ("#fcfcfb", "#0b0b0b", "#52514e",
                                         "#898781", "#e1e0d9", "#c3c2b7")
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False


def style(ax, grid_axis="y"):
    ax.set_facecolor(SURFACE)
    ax.grid(True, axis=grid_axis, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(AXIS)
    ax.tick_params(colors=MUTED, labelsize=9)


# 그래프 1 — 에폭별 곡선 (줄마다 색 하나, 점선 = 기준선, 표시 = 얼리스탑이 돌려준 에폭)
fig, axes = plt.subplots(len(fws), 2, figsize=(11, 3.8 * len(fws) + 1.0), dpi=150,
                         squeeze=False, facecolor=SURFACE)
for i, fw in enumerate(fws):
    for j, (name, key) in enumerate((("val loss", "val_loss"), ("val acc (%)", "val_acc"))):
        ax = axes[i][j]
        style(ax, "y")
        for a in arms:
            group = curves.get((fw, a))
            if not group:
                continue
            (color, mark), ls = STYLE[a], ("--" if a == BASE_LAB else "-")
            many = len(group) > 1
            for c in group.values():
                y = c[key]
                ax.plot(np.arange(1, len(y) + 1), y, color=color, ls=ls, lw=0.9 if many else 2.4,
                        alpha=0.35 if many else 1.0, zorder=2)
                if c["best"] > 0:
                    ax.plot([c["best"]], [y[c["best"] - 1]], ls="none", marker=mark, ms=7,
                            mfc=color, mec=SURFACE, mew=1.0, zorder=5)
            n = min(len(c[key]) for c in group.values())
            if many and n > 0:
                ax.plot(np.arange(1, n + 1), np.mean([c[key][:n] for c in group.values()], axis=0),
                        color=color, ls=ls, lw=2.4, zorder=3)
        ax.set_xlabel("epoch", fontsize=9, color=INK2)
        ax.set_title(f"{FW[fw]} · {name}", fontsize=10.5, color=INK, loc="left")
handles = [Line2D([0], [0], color=STYLE[a][0], ls="--" if a == BASE_LAB else "-", lw=2.4,
                  marker=STYLE[a][1], ms=7, mec=SURFACE, label=a + (" (base)" if a == BASE_LAB else ""))
           for a in arms]
fig.text(0.005, 0.99, f"Learning curves, early stopping (cap {EPOCH_CAP}, patience {PATIENCE})  -  "
         "marker = epoch of the returned model", fontsize=12, color=INK, va="top")
fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.0, 0.955),
           ncol=len(handles), frameon=False, fontsize=9, labelcolor=INK2)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(f"{FIG_PREFIX}_curves.png", facecolor=SURFACE)
plt.show()
print(f"[그래프] {FIG_PREFIX}_curves.png")

# 그래프 2 — 줄별 결과 (점 하나 = 시드 하나)
PANELS = [("val_acc", "val acc (%) of the returned model", 1.0),
          ("val_loss", "best val loss", 0.03),
          ("gap", "train (clean) - val acc gap (%p)", 1.0)]
fig, axes = plt.subplots(len(fws), 3, figsize=(13, 3.6 * len(fws) + 0.9), dpi=150,
                         squeeze=False, facecolor=SURFACE)
for i, fw in enumerate(fws):
    for j, (key, name, min_pad) in enumerate(PANELS):
        ax = axes[i][j]
        style(ax)
        seen = []
        for x, a in enumerate(arms):
            pts = [(s, m[key]) for s, m in sorted(runs.get((fw, a), {}).items()) if np.isfinite(m[key])]
            if not pts:
                continue
            color = STYLE[a][0]
            ys = np.array([p[1] for p in pts])
            seen += list(ys)
            ax.vlines(x, ys.min(), ys.max(), color=color, lw=1.6, alpha=0.45, zorder=2)
            ax.plot([x - 0.3, x + 0.3], [ys.mean()] * 2, color=color, lw=2.4,
                    solid_capstyle="butt", zorder=3)
            ax.plot([x + 0.2 * (s - 1) for s, _ in pts], ys, ls="none", marker="o", ms=6.5,
                    mfc=color, mec=SURFACE, mew=1.1, zorder=4)
        if seen:
            lo, hi = min(seen), max(seen)
            margin = max(min_pad, (hi - lo) * 0.2)
            ax.set_ylim(lo - margin, hi + margin)
        ax.set_xlim(-0.6, len(arms) - 0.4)
        ax.set_xticks(range(len(arms)))
        ax.set_xticklabels([a + (" (base)" if a == BASE_LAB else "") for a in arms])
        ax.set_xlabel(X_LABEL, fontsize=9, color=INK2)
        ax.set_title(f"{FW[fw]} · {name}", fontsize=10.5, color=INK, loc="left")
handles = [Line2D([0], [0], color=INK2, lw=2.4, label="mean"),
           Line2D([0], [0], marker="o", ls="none", ms=6.5, mfc=INK2, mec=SURFACE, mew=1.1,
                  label="each seed (left to right: 0, 1, 2)")]
fig.text(0.005, 0.99, FIG_TITLE, fontsize=12, color=INK, va="top")
fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.0, 0.955),
           ncol=len(handles), frameon=False, fontsize=9, labelcolor=INK2)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(f"{FIG_PREFIX}_seeds.png", facecolor=SURFACE)
plt.show()
print(f"[그래프] {FIG_PREFIX}_seeds.png")